# Как работает интернет

1. Задача: молекулы из PubChem
2. Что произошло, когда мы открыли ссылку?
3. Страница для человека и API для программы
4. Как устроена ссылка
5. Как ссылка превращается в адрес: DNS
6. Как передаётся информация между клиентом и сервером
7. Как проектируют взаимодействие: REST
8. Разведка: PubChem
9. Формат данных: JSON
10. Практика: `requests`
11. Дополнительно: как всё устроено подробнее

> Ячейки, которым нужен интернет, помечены комментарием `# нужен интернет`.

# Задача: молекулы из PubChem

Вася хочет посмотреть на одной карте свои молекулы и известные препараты: аспирин, ибупрофен, кофеин, парацетамол. Сейчас он открывает [PubChem](https://pubchem.ncbi.nlm.nih.gov/) в браузере, ищет каждое вещество, копирует SMILES и вставляет в CSV руками. Для четырёх молекул терпимо, для сорока – уже нет.

**Хочется:** передать пайплайну список названий, а он сам сходит на PubChem, возьмёт SMILES и построит карту:

```
uv run python main.py pubchem aspirin ibuprofen caffeine paracetamol
```

**Требования:**
* новый источник молекул `PubChemSource`, который по списку названий получает SMILES с PubChem;
* `mol_id` – название вещества, как его ввёл пользователь;


## Посмотрим на PubChem

Откройте страницу аспирина: [pubchem.ncbi.nlm.nih.gov/compound/Aspirin](https://pubchem.ncbi.nlm.nih.gov/compound/Aspirin). SMILES лежит в разделе **Names and Identifiers → Computed Descriptors**. Так сейчас и работает Вася: находит вещество и копирует SMILES руками.

# Что произошло, когда мы открыли ссылку?

За одним кликом стоит целая цепочка:

1. браузер разобрал ссылку: каким протоколом идти, на какую машину, за каким ресурсом – раздел «Как устроена ссылка»;
2. узнал IP-адрес машины по её имени – раздел «DNS»;
3. установил связь с сервером и отправил ему запрос – раздел «Как передаётся информация»;
4. программа на сервере разобрала запрос, подготовила ответ и отправила его обратно – об этом ниже;
5. браузер получил HTML, дозапросил всё, что нужно для страницы, и нарисовал её – тоже ниже.

![Что происходит, когда открываем ссылку: браузер, DNS и сервер PubChem](images/open_link.svg)

Начнём с конца: кто и как ответил на наш запрос.


**Сервер** – это обычная программа, которая запущена и **ждёт событий**: входящих запросов. Для каждого запроса она:
1. принимает соединение и читает запрос;
2. по пути из запроса (`/compound/aspirin`) находит, какая функция за него отвечает;
3. вызывает эту функцию и отправляет клиенту её результат как ответ;
4. возвращается к ожиданию следующего запроса.

Мы не пишем этот цикл сами: его даёт веб-фреймворк. Мы только регистрируем **обработчики событий** – функции, которые вызываются на определённые запросы. Сделаем маленький «PubChem» на Flask:

In [4]:
from flask import Flask, jsonify

app = Flask(__name__)

COMPOUNDS = {
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
}


@app.route("/compound/<name>")  # регистрируем обработчик для путей вида /compound/...
def get_compound(name: str):
    if name not in COMPOUNDS:
        return jsonify({"error": f"compound {name!r} not found"}), 404
    return jsonify({"name": name, "smiles": COMPOUNDS[name]})


print(app.url_map)  # таблица «путь -> обработчик», которую собрал декоратор

Map([<Rule '/static/<filename>' (OPTIONS, GET, HEAD) -> static>,
 <Rule '/compound/<name>' (OPTIONS, GET, HEAD) -> get_compound>])


In [6]:
client = app.test_client()  # отправляет запросы прямо в приложение, без сети

for name in ["aspirin", "ibuprofen"]:
    response = client.get(f"/compound/{name}")
    print(response.get_json())

{'name': 'aspirin', 'smiles': 'CC(=O)Oc1ccccc1C(=O)O'}
{'error': "compound 'ibuprofen' not found"}


## Что может отдавать сервер

В ответе сервера – просто байты: страница, данные, таблица, файл, картинка, архив. Что именно пришло, сервер сообщает в заголовке **`Content-Type`**, а клиент по нему решает, что с ответом делать: браузер рисует страницу или показывает картинку, программа разбирает данные или сохраняет файл.

Часто один и тот же ресурс можно получить в разных форматах – это решает тот, кто проектировал сервер.

## Что на самом деле пришло в браузер: HTML, CSS, JavaScript и JSON

Страница, которую мы видим, собирается браузером из нескольких частей. Каждая приходит отдельным запросом:

| Что | Зачем | Пример |
|---|---|---|
| **HTML** | структура и содержимое: заголовки, абзацы, таблицы, ссылки | `<h3>SMILES</h3><div class="smiles">CC(=O)O...</div>` |
| **CSS** | внешний вид: цвета, шрифты, расположение | `.smiles { font-family: monospace; color: gray; }` |
| **JavaScript** | поведение: кнопки, поиск, подгрузка данных после открытия страницы | `fetch("/rest/pug/compound/...")` |
| **JSON** | данные для программ, без оформления | `{"CID": 2244, "SMILES": "CC(=O)O..."}` |

### Как браузер строит страницу

1. получает HTML и разбирает его в дерево Document Object Model;
2. находит в нём ссылки на CSS, JavaScript и картинки и запрашивает каждый файл отдельно;
3. применяет CSS к узлам дерева: какого цвета, какого размера, где на экране;
4. рисует страницу;
5. выполняет JavaScript. Он может менять дерево – например, запросить данные в JSON и добавить новые узлы. Тогда браузер перерисовывает изменившуюся часть.

Современные сайты часто устроены так: браузер получает почти пустой HTML, а уже потом JavaScript на странице сам запрашивает данные в JSON и вставляет их в нужные места. Так работает и PubChem: во вкладке Network инструментов разработчика (`F12`) видно, как страница после загрузки ходит за данными.

### Как устроен HTML

HTML состоит из **элементов**. Элемент – это:
* открывающий тег `<a ...>`;
* содержимое – текст или другие элементы;
* закрывающий тег `</a>`.

У открывающего тега бывают **атрибуты** – дополнительные свойства вида `имя="значение"`:

```html
<a href="https://pubchem.ncbi.nlm.nih.gov">PubChem</a>
```

Здесь `a` – тег ссылки, `href` – атрибут с адресом, `PubChem` – содержимое. Чаще всего встречаются атрибуты `id` (уникальное имя элемента) и `class` (группа элементов, к которой применяются одни и те же стили).

### Вложенность и дерево

Элементы вкладываются друг в друга, поэтому вся страница – это **дерево**:

```html
<html>
  <head>
    <title>Aspirin</title>
  </head>
  <body>
    <h1>Aspirin</h1>
    <section id="descriptors">
      <h3>SMILES</h3>
      <div class="smiles">CC(=O)OC1=CC=CC=C1C(=O)O</div>
    </section>
  </body>
</html>
```

```
html
├── head
│   └── title: "Aspirin"
└── body
    ├── h1: "Aspirin"
    └── section (id="descriptors")
        ├── h3: "SMILES"
        └── div (class="smiles"): "CC(=O)OC1=CC=CC=C1C(=O)O"
```

Корень – `html`, у него два ребёнка – `head` (служебное: заголовок вкладки, подключение стилей и скриптов) и `body` (то, что видно на странице). Это дерево браузер называет **DOM** (Document Object Model).

**HTML и DOM – не одно и то же.** HTML – это текст, который пришёл с сервера. DOM – то, что браузер из него **построил**: дерево объектов в памяти, у каждого узла есть родитель, дети, атрибуты и текст. Похоже на SMILES и `Chem.Mol`: `"CC(=O)O"` – просто строка, а `Chem.MolFromSmiles` превращает её в граф атомов, у которого можно что-то спросить и что-то поменять.

JavaScript работает именно с DOM: находит узлы, меняет их, добавляет новые – и браузер перерисовывает страницу. Поэтому DOM может отличаться от пришедшего HTML. Это видно прямо в браузере: `Ctrl+U` показывает исходный HTML от сервера, а `F12` → Elements – текущий DOM, уже с тем, что дорисовал JavaScript.

**Парсинг страницы** – это поиск по такому дереву: «найди `div` с классом `smiles` внутри `section`». Именно это и делает BeautifulSoup:

```python
from bs4 import BeautifulSoup

soup = BeautifulSoup(PAGE, "html.parser")
smiles = soup.find("section", id="descriptors").find("div", class_="smiles").text
```

Так делать можно, но есть проблемы:
* вёрстку меняют без предупреждения, и парсер молча ломается;
* если данные подгружает JavaScript, в скачанном HTML их может не оказаться вовсе: Python не выполняет JavaScript, как браузер;
* сайты защищаются от роботов. Посмотрим, что вернёт PubChem, если попросить у него страницу из Python:

In [1]:
import requests

response = requests.get("https://pubchem.ncbi.nlm.nih.gov/compound/Aspirin", timeout=10)  # нужен интернет
print(response.status_code, response.headers["Content-Type"])
print("SMILES аспирина в HTML:", "CC(=O)OC1=CC=CC=C1C(=O)O" in response.text)
print(response.text[:200])

200 text/html; charset=utf-8
SMILES аспирина в HTML: False
<!doctype html><html lang="en-US" dir="ltr"><head><base href="https://www.google.com/recaptcha/challengepage/"><link rel="preconnect" href="//www.gstatic.com"><meta name="referrer" content="origin"><s


# Страница для человека и API для программы

Код ответа `200`, но SMILES в нём нет: страница сделана для человека и браузера, а не для программы. Для программ у сайтов обычно есть отдельный вход.

Один и тот же сервер может отвечать по-разному в зависимости от пути. Для человека – страницей, для программы – через **API** (Application Programming Interface). Это тот же «интерфейс», что и у классов на прошлой паре: договорённость, какие запросы можно делать и что придёт в ответ. Только «методы» вызываются по сети.

| | Страница | API |
|---|---|---|
| Для кого | человек в браузере | программа |
| Что приходит | HTML, CSS, JavaScript; данные спрятаны внутри вёрстки | только данные: JSON, CSV, текст |
| Как меняется | дизайн меняют когда угодно | формат стабилен и описан в документации |
| У PubChem | [/compound/Aspirin](https://pubchem.ncbi.nlm.nih.gov/compound/Aspirin) | [/rest/pug/compound/name/aspirin/property/SMILES/JSON](https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/SMILES/JSON) |

Откройте вторую ссылку в браузере: вместо страницы – только JSON со SMILES. Наш Flask-сервис из начала тоже был API: на `/compound/aspirin` он отдавал не страницу, а JSON.

Чтобы пользоваться API из программы, нужно понимать, как устроен адрес запроса, как запрос доходит до сервера и о чём клиент с сервером договариваются. Об этом – следующие разделы.

# Как устроена ссылка

```
https://www.ebi.ac.uk:443/chembl/api/data/molecule/search.json?q=aspirin&limit=1
```

* **схема** `https` – каким протоколом говорить (`https`, `http`, `ftp`);
* **хост** `www.ebi.ac.uk` – имя машины. Компьютеры общаются по IP-адресам, поэтому имя сначала надо превратить в адрес;
* **порт** `443` – номер «двери» на машине; у `https` он по умолчанию, поэтому его обычно не пишут;
* **путь** `/chembl/api/data/molecule/search.json` – какой ресурс нужен. Это не путь к файлу на сервере: сервер сам решает, какой обработчик за него отвечает, как наш `@app.route`;
* **query string** `?q=aspirin&limit=1` – дополнительные параметры в виде `ключ=значение`, через `&`.

`requests` сам собирает query string из словаря и кодирует спецсимволы:

In [2]:
import requests

request = requests.Request("GET", "https://www.ebi.ac.uk/chembl/api/data/molecule/search.json",
                           params={"q": "acetylsalicylic acid", "limit": 1}).prepare()
print(request.url)  # пробел превратился в +, параметры склеены через &

https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q=acetylsalicylic+acid&limit=1


# Как ссылка превращается в адрес: DNS

Перед любым запросом нужно узнать IP-адрес хоста. Этим занимается **DNS** – распределённая «телефонная книга» интернета. Ответ ищется по цепочке, и каждый шаг кэширует результат:

1. **кэш** браузера или операционной системы: может, адрес уже известен;
2. **резолвер** провайдера или публичный (например, 8.8.8.8), если в кэше нет;
3. резолвер спрашивает **корневой сервер**: «кто отвечает за `.gov`?»;
4. потом **сервер зоны `.gov`**: «кто отвечает за `nih.gov`?»;
5. потом **сервер `nih.gov`**: «какой адрес у `pubchem.ncbi.nlm.nih.gov`?» – и получает IP.

Из Python это одна строка. В терминале то же самое покажет `nslookup pubchem.ncbi.nlm.nih.gov`.

In [ ]:
import socket

print(socket.gethostbyname("pubchem.ncbi.nlm.nih.gov"))  # нужен интернет
print(socket.gethostbyname("localhost"))  # своя машина: 127.0.0.1

# Как передаётся информация между клиентом и сервером

Адрес известен, нам нужно организовать процесс общения сервера с клиентом. Для этого нам нужны:

1. **TCP-соединение** на порт сервера (443): надёжный канал, байты приходят целыми и по порядку;
2. **TLS** – шифрование и проверка сертификата: мы точно говорим с PubChem, и никто посередине не читает трафик. Это буква S в HTTPS;
3. **HTTP** – сам разговор. Клиент отправляет **запрос**, сервер возвращает **ответ**. Оба – обычный текст:

```
GET /rest/pug/compound/name/aspirin/property/SMILES/JSON HTTP/1.1 <- метод, путь, версия
Host: pubchem.ncbi.nlm.nih.gov <- заголовки
User-Agent: python-requests/2.34

HTTP/1.1 200 OK <- статус
Content-Type: application/json <- заголовки
<- пустая строка
{"PropertyTable": {"Properties": [{"CID": 2244, "SMILES": "..."}]}} <- тело
```

`requests` делает всё это за нас, а в объекте ответа видно обе стороны разговора:

In [9]:
response = requests.get(  # нужен интернет
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/SMILES/JSON",
    timeout=10,
)
print("запрос:", response.request.method, response.request.url)
print("заголовки запроса:", dict(response.request.headers))
print("статус:", response.status_code, response.reason)
print("заголовки ответа:", response.headers["Content-Type"])
print("тело:", response.text[:120])

запрос: GET https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/SMILES/JSON
заголовки запроса: {'User-Agent': 'python-requests/2.34.2', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}
статус: 200 OK
заголовки ответа: application/json
тело: {
  "PropertyTable": {
    "Properties": [
      {
        "CID": 2244,
        "SMILES": "CC(=O)OC1=CC=CC=C1C(=O)O"
   


Весь этот путь можно увидеть одной командой в терминале:

```
curl -v https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/SMILES/JSON
```

В выводе по порядку:
* `Trying <IP-адрес>:443...` – DNS отдал адрес, открываем TCP-соединение на порт 443;
* `TLS handshake` – договариваемся о шифровании и проверяем сертификат;
* строки с `>` – наш HTTP-запрос;
* строки с `<` – заголовки ответа сервера;
* тело ответа – JSON со SMILES аспирина.

# Как проектируют взаимодействие: REST

**REST** – самый распространённый стиль проектирования API поверх HTTP.

HTTP и REST – не одно и то же:
* **HTTP** – **протокол**: правила, по которым клиент и сервер обмениваются сообщениями;
* **REST** – **стиль проектирования** поверх HTTP: соглашения, как разложить API по путям и методам. HTTP прекрасно работает и без REST: обычные страницы, скачивание файлов, другие стили API.

Основные соглашения REST:

* **ресурс в URL**: путь описывает «что», а не «что сделать» – `/compound/aspirin`, а не `/getCompound?name=aspirin`;
* **метод – глагол**:

| Метод | Смысл | Тело запроса |
|---|---|---|
| `GET` | получить ресурс | нет: параметры в URL |
| `POST` | создать новый | да |
| `PUT` / `PATCH` | заменить / изменить | да |
| `DELETE` | удалить | обычно нет |
| `QUERY` | получить, передав условия запроса в теле – новый метод | да: условия запроса |

**Зачем `QUERY`.** У `GET` все параметры живут в URL. Если условия поиска большие – например, список из тысячи названий или сложный фильтр, – URL упирается в ограничения длины, а длинные URL ещё и оседают в логах серверов. Раньше это обходили так:
* **`GET` с телом запроса** – технически отправить можно, но стандарт HTTP не определяет, что такое тело значит, и прокси, кэши и библиотеки могут его отбросить. Некоторые API так делают (например, поиск в Elasticsearch), но это ненадёжно;
* **`POST` для поиска** – работает везде, но `POST` по смыслу «создай что-то». Его нельзя безопасно повторить и закэшировать: клиент и прокси не знают, что запрос только читает данные.

Метод **`QUERY`** закрывает этот пробел: тело запроса с условиями, как у `POST`, но смысл как у `GET` – только прочитать, без изменений, можно безопасно повторять и кэшировать. Его предложила рабочая группа IETF по HTTP; поддержка в серверах, прокси и библиотеках только появляется, так что на практике пока чаще встретите `POST` для поиска.

* **код ответа – результат**:

| Класс | Смысл | Частые коды |
|---|---|---|
| 2xx | успех | `200 OK`, `201 Created` |
| 3xx | ресурс в другом месте | `301`, `302` |
| 4xx | **ошибка клиента** – проблема в нашем запросе | `400 Bad Request`, `404 Not Found`, `429 Too Many Requests` |
| 5xx | **ошибка сервера** | `500 Internal Server Error`, `503 Service Unavailable` |

* **сервер не помнит прошлых запросов**: каждый запрос самодостаточен.

Правило: **4xx – разбираемся со своим запросом, 5xx – ждём и пробуем позже.**

Наш Flask-сервис из начала – маленький REST API: ресурс `/compound/<name>`, метод `GET`, коды `200` и `404`. PubChem PUG REST устроен так же, только ресурсов у него тысячи. Альтернативы (RPC, GraphQL, очереди сообщений) – в дополнительной части.

# Разведка: PubChem

Прежде чем писать источник, посмотрим, что именно отвечает PubChem. Формат запроса – в документации PUG REST. Путь читается почти как фраза:

```
https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/SMILES,MolecularWeight/JSON
```

* `/rest/pug` – вход в API (а не на страницы для людей);
* `/compound` – какие записи ищем: соединения;
* `/name/aspirin` – как ищем: по названию `aspirin`;
* `/property/SMILES,MolecularWeight` – что хотим получить: свойства через запятую;
* `/JSON` – в каком формате прислать ответ. Замените на `CSV` или `TXT` – получите те же данные в другом виде.

In [ ]:
import json

BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

response = requests.get(f"{BASE_URL}/compound/name/aspirin/property/SMILES,MolecularWeight/JSON", timeout=10)  # нужен интернет
print(response.status_code, response.headers["Content-Type"])
data = response.json()
print(json.dumps(data, indent=2))

In [ ]:
print(data["PropertyTable"]["Properties"][0]["SMILES"])  # путь до SMILES в ответе

А что, если вещества нет? Смотрим на код и тело ответа:

In [ ]:
response = requests.get(f"{BASE_URL}/compound/name/not_a_real_compound/property/SMILES/JSON", timeout=10)  # нужен интернет
print(response.status_code)
print(response.json())

Что узнали:
* SMILES лежит в `["PropertyTable"]["Properties"][0]["SMILES"]`;
* неизвестное название – код `404` и `Fault` в теле вместо данных;
* по правилам PubChem – не больше **5 запросов в секунду**; при перегрузке сервер отвечает `503`.

# Формат данных: JSON

**JSON** (JavaScript Object Notation) — текстовый формат «ключ — значение». Самый частый формат ответов API.

```json
{
  "CID": 2244,
  "MolecularFormula": "C9H8O4",
  "IsDrug": true,
  "Synonyms": ["aspirin", "acetylsalicylic acid"],
  "Patent": null
}
```

Похож на словарь Python, но есть отличия:
* строки только в **двойных** кавычках;
* `true` / `false` / `null` вместо `True` / `False` / `None`;
* ключи объекта — **всегда строки**;
* нет дат, множеств, кортежей; `NaN` и `Infinity` стандартом не разрешены;
* числа — просто числа: без различия `int` и `float` на уровне формата.

## Модуль `json`

| Функция | Что делает |
|---|---|
| `json.loads(text)` | строка → объект Python (s — string) |
| `json.dumps(obj)` | объект Python → строка |
| `json.load(file)` | файл → объект Python |
| `json.dump(obj, file)` | объект Python → файл |

In [ ]:
import json

text = '{"CID": 2244, "MolecularFormula": "C9H8O4", "IsDrug": true, "Patent": null}'
data = json.loads(text)
print(data, type(data))

print(json.dumps(data))  # True -> true, None -> null
print(json.dumps({"name": "аспирин"}, ensure_ascii=False, indent=2))  # для людей: кириллица и отступы

## JSON → датакласс

Словарь из ответа API — это та же проблема, что и словарь-молекула на первой паре: поля нигде не описаны, опечатка в ключе всплывёт только при запуске. Поэтому ответ стоит сразу превратить в датакласс:

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class CompoundInfo:
    cid: int
    formula: str
    molecular_weight: float

    @classmethod
    def from_json(cls, data: dict) -> "CompoundInfo":
        return cls(
            cid=data["CID"],
            formula=data["MolecularFormula"],
            molecular_weight=float(data["MolecularWeight"]),  # API может отдать число строкой
        )


response_data = {"CID": 2244, "MolecularFormula": "C9H8O4", "MolecularWeight": "180.16"}
print(CompoundInfo.from_json(response_data))

Если полей много или их надо строго проверять, для этого есть библиотеки валидации, например **pydantic**: описываете модель, и она сама проверяет типы и приводит значения.

# Практика: `requests`

## Объект `Response`

| Атрибут | Что внутри |
|---|---|
| `status_code` | статус-код |
| `headers` | заголовки ответа (словарь без учёта регистра ключей) |
| `text` | тело как строка |
| `content` | тело как байты: для картинок, архивов, SDF |
| `json()` | тело, разобранное из JSON |
| `url` | итоговый URL (после редиректов и с query string) |
| `request` | что именно мы отправили: URL, заголовки |

## Параметры query string: `params`

Параметры не нужно склеивать в URL руками: `requests` сам соберёт query string и правильно закодирует пробелы и спецсимволы.

In [ ]:
response = requests.get(  # нужен интернет
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
    params={"db": "pccompound", "term": "acetylsalicylic acid", "retmode": "json"},
    timeout=10,
)
print(response.url)  # пробел закодирован, параметры склеены через &
print(response.json())

## Ошибки: сервер ответил «нет» и «до сервера не дошли» — разные вещи

* Сервер **ответил**, но с ошибкой (404, 500): исключения нет, есть ответ с кодом.
* До сервера **не дошли** (опечатка в хосте, нет сети, таймаут): исключение.

In [ ]:
response = requests.get(  # нужен интернет
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/not_a_real_compound_name/JSON",
    timeout=10,
)
print(response.status_code)  # сервер ответил: такого соединения нет

try:
    requests.get("https://pubchem.ncbi.nlm.nih.goooov", timeout=10)
except requests.ConnectionError as error:
    print("до сервера не дошли:", type(error).__name__)

## Best practices

### 1. Всегда `timeout`

**У `requests` нет таймаута по умолчанию.** Если сервер повиснет, ваш скрипт будет ждать вечно. Таймаут можно задать одним числом или парой «на соединение, на чтение».

### 2. Проверяем код ответа

Без проверки ответ 404 или 500 молча пойдёт дальше по коду, например в `.json()`, и упадёт там с непонятной ошибкой. Поэтому сразу после запроса смотрим на `status_code`. Чтобы не писать «магические» `404` и `200`, в стандартной библиотеке есть `HTTPStatus`. Короткий вариант для случаев, когда любой отказ – ошибка: `response.raise_for_status()` превращает 4xx и 5xx в исключение `HTTPError`.

In [ ]:
from http import HTTPStatus


def get_formula(name: str) -> str | None:
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/MolecularFormula/JSON"
    response = requests.get(url, timeout=(3, 10))  # 3 с на соединение, 10 с на чтение
    if response.status_code == HTTPStatus.NOT_FOUND:
        return None  # такого вещества нет – это ожидаемый исход
    if response.status_code != HTTPStatus.OK:
        raise RuntimeError(f"PubChem returned status {response.status_code} for {name!r}")
    return response.json()["PropertyTable"]["Properties"][0]["MolecularFormula"]


print(get_formula("caffeine"))  # нужен интернет
print(get_formula("not_a_real_compound_name"))

### 3. `Session` для серии запросов

Каждый `requests.get` заново открывает соединение: DNS, TCP, TLS. `Session` переиспользует соединение и хранит общие настройки (заголовки, параметры) для всех запросов.

In [ ]:
with requests.Session() as session:  # нужен интернет
    session.headers["Accept"] = "application/json"
    for name in ["aspirin", "caffeine", "ibuprofen"]:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/MolecularWeight/JSON"
        response = session.get(url, timeout=10)
        response.raise_for_status()
        print(name, response.json()["PropertyTable"]["Properties"][0]["MolecularWeight"])

### 4. Никогда `verify=False`

`verify=False` отключает проверку сертификата: вы больше не знаете, с тем ли сервером разговариваете. Если `requests` ругается на сертификат, это проблема, которую надо решить, а не заглушить.

### 5. Ключи API — не в коде

Многие API требуют ключ (токен). Его **нельзя** писать в коде и коммитить в git: репозиторий может оказаться публичным, а из истории git ключ уже не удалить. Ключ берут из переменной окружения:

```python
import os

api_key = os.environ["SOME_API_KEY"]
response = requests.get(url, headers={"Authorization": f"Bearer {api_key}"}, timeout=10)
```

### 6. Читаем правила использования API

У публичных баз есть правила: сколько запросов в секунду можно делать, нужно ли представляться в `User-Agent`, можно ли скачивать базу целиком. PubChem, например, просит не больше 5 запросов в секунду, а при перегрузке отвечает `503`. Нарушителей временно блокируют.

## Слишком много запросов: 429 и ретраи

Если запросов слишком много, сервер отвечает **`429 Too Many Requests`**, часто с заголовком **`Retry-After`**: через сколько секунд можно повторить.

Что делать:
* не отправлять запросы чаще, чем разрешено, — **ограничитель частоты** (rate limiter);
* при 429 и 5xx подождать и повторить: каждый раз ждать дольше (**exponential backoff**);
* повторять можно только **идемпотентные** запросы.

### Домашнее задание

Написать **rate limiter**: класс и/или декоратор, который не даёт вызывать функцию чаще N раз в секунду. Плюс обработка ответа 429 с учётом заголовка `Retry-After`.

# Дополнительно: как всё устроено подробнее

Ниже – более подробный материал: уровни сетевого взаимодействия, протоколы, стили API, JSON и практика `requests`. Погружаемся, если останется время.

# Что нужно, чтобы получить информацию с другой машины

В пайплайне у нас уже есть два источника молекул: файл и REINVENT. Третий источник — база в интернете, например PubChem. Чтобы им воспользоваться, нужно понимать, что происходит между `requests.get(...)` и ответом.

Чтобы одна программа получила данные от другой программы на другой машине, нужно договориться о трёх вещах:

| Уровень | Вопрос | Что отвечает | Аналогия: звонок в колл-центр |
|---|---|---|---|
| **1. Найти машину и соединиться** | куда и как доставить байты? | DNS, IP, порт, TCP/UDP, TLS | найти номер в справочнике, набрать, дождаться ответа |
| **2. Протокол** | на каком языке говорим и кто когда говорит? | HTTP, FTP, SSH, SMTP, WebSocket... | договориться, что говорим по-русски: «алло», вопрос, ответ, «до свидания» |
| **3. Интерфейс (API)** | что именно можно спросить и в каком виде придёт ответ? | стиль API (REST, RPC, GraphQL...), формат данных (JSON, XML), документация | меню: «нажмите 1, чтобы узнать баланс» |

Уровни 1–2 — это упрощённая модель TCP/IP. Уровня 3 в классических сетевых моделях нет: это договорённости между разработчиками. Но именно с ним вы будете работать каждый день.

# Уровень 1: найти машину и соединиться

## Клиент и сервер

**Клиент** — программа, которая отправляет запрос. **Сервер** — программа, которая его принимает и отвечает. Это **программы, а не железо**: клиент и сервер спокойно могут работать на одном компьютере. Когда вы открываете сайт, клиент — это браузер. Когда вызываете `requests.get`, клиент — ваш Python-скрипт.

## Адрес: URL

```
https://pubchem.ncbi.nlm.nih.gov:443/rest/pug/compound/name/aspirin/JSON?record_type=2d
└─┬─┘   └──────────┬──────────┘ └┬┘ └──────────────┬─────────────────┘ └─────┬──────┘
схема           хост          порт              путь                  query string
```

* **схема** — каким протоколом пользоваться (`https`, `http`, `ftp`, `wss`...);
* **хост** — имя машины. Компьютеры работают с IP-адресами, поэтому имя сначала превращается в адрес (это DNS);
* **порт** — номер «двери» на машине. Один сервер может одновременно принимать HTTPS на 443 и SSH на 22. Обычно порт не пишут: у каждой схемы есть порт по умолчанию;
* **путь** — какой ресурс нужен. Это не путь в файловой системе сервера: сервер сам решает, что по нему отдать;
* **query string** — дополнительные параметры `?key1=value1&key2=value2`.

## DNS: имя → адрес

Первый шаг любого запроса — спросить у DNS-сервера, какой IP-адрес у хоста. Это можно сделать прямо из Python:

In [ ]:
import socket

print(socket.gethostbyname("pubchem.ncbi.nlm.nih.gov"))  # нужен интернет
print(socket.gethostbyname("localhost"))

## Доставка байтов: транспортные протоколы

| Протокол | Что делает | Важно знать |
|---|---|---|
| **IP** | доставляет пакеты по адресу | IPv4 (`93.184.216.34`) и IPv6; гарантий доставки не даёт |
| **TCP** | надёжный канал поверх IP: порядок, повторная отправка потерь | перед началом — «рукопожатие»; на нём HTTP/1.1, HTTP/2, SSH, FTP, SMTP |
| **UDP** | просто отправить пакет, без гарантий | быстрее, без рукопожатия; на нём DNS, видеозвонки, игры |
| **QUIC** | надёжный канал поверх UDP со встроенным шифрованием | быстрее устанавливает соединение; на нём HTTP/3 |
| **TLS** | шифрование и проверка, что сервер — тот, за кого себя выдаёт | буква S в HTTPS; работает на сертификатах |
| **DNS** | имя → IP-адрес | обычно поверх UDP, порт 53 |

Порты наизусть знать не нужно, но три стоит запомнить: **80** (HTTP), **443** (HTTPS), **22** (SSH).

## Весь путь в одном выводе: `curl -v`

Запустите в терминале:

```
curl -v https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/MolecularFormula/JSON
```

В выводе видно всё по порядку:
* `Trying <IP-адрес>:443...` — DNS отдал адрес, открываем TCP-соединение на порт 443;
* `TLS handshake` — договариваемся о шифровании, проверяем сертификат;
* строки с `>` — наш HTTP-запрос;
* строки с `<` — заголовки ответа сервера;
* тело ответа — JSON с формулой аспирина.

# Уровень 2: протокол

Соединение есть. Теперь нужно договориться, **о чём и как мы говорим**. Для веба и большинства API это **HTTP**.

## HTTP — это просто текст

**Запрос** состоит из трёх частей:
1. стартовая строка: **метод**, путь, версия протокола;
2. **заголовки** — пары `Ключ: значение`;
3. **тело** (необязательно) — данные, которые мы отправляем.

```
GET /rest/pug/compound/name/aspirin/JSON HTTP/1.1
Host: pubchem.ncbi.nlm.nih.gov
User-Agent: python-requests/2.32
Accept: application/json
```

**Ответ** устроен так же, только вместо метода — **статус-код**:

```
HTTP/1.1 200 OK
Content-Type: application/json
Content-Length: 4521

{"PC_Compounds": [...]}
```

Чтобы убедиться, что это действительно просто текст, отправим запрос руками через сокет: откроем TCP-соединение (уровень 1) и напишем в него HTTP (уровень 2).

In [ ]:
import socket

request = (
    "GET / HTTP/1.1\r\n"
    "Host: example.com\r\n"
    "Connection: close\r\n"
    "\r\n"  # пустая строка: заголовки закончились
)

with socket.create_connection(("example.com", 80), timeout=10) as connection:  # нужен интернет
    connection.sendall(request.encode())
    response = b""
    while chunk := connection.recv(4096):
        response += chunk

print(response.decode(errors="replace")[:600])

Так писать, конечно, никто не будет: нужно самим собирать заголовки, разбирать ответ, следить за кодировками и шифрованием. Поэтому над сокетами строят библиотеки:

| Уровень | Что | Как выглядит работа |
|---|---|---|
| `socket` | сырое TCP-соединение | сами пишем текст запроса и разбираем ответ |
| `urllib.request` | HTTP-клиент из стандартной библиотеки | работает без установки, но многословен: кодировки, ошибки, JSON — всё вручную |
| `urllib3` | пул соединений, ретраи | на нём построен `requests` |
| **`requests`** | HTTP «для людей» | одна строка на запрос — дальше работаем только с ним |
| `httpx` | как `requests`, плюс async и HTTP/2 | пригодится, когда дойдём до параллельности |

## Методы

**Метод** говорит серверу, что мы хотим сделать:

| Метод | Смысл | Тело запроса |
|---|---|---|
| **GET** | получить данные | нет; параметры — в query string |
| **POST** | отправить данные, создать что-то | да |
| **PUT** | заменить ресурс целиком | да |
| **PATCH** | изменить ресурс частично | да |
| **DELETE** | удалить ресурс | обычно нет |

Сервер может трактовать методы как угодно, но это общепринятые соглашения.

## Статус-коды

Трёхзначное число в ответе. Важнее всего **первая цифра**:

| Класс | Смысл | Самые частые |
|---|---|---|
| **1xx** | информационные | `101 Switching Protocols` (переход на WebSocket) |
| **2xx** | успех | `200 OK`, `201 Created`, `204 No Content` |
| **3xx** | перенаправление: ресурс в другом месте | `301 Moved Permanently`, `302 Found` |
| **4xx** | **ошибка клиента** — проблема в нашем запросе | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `429 Too Many Requests` |
| **5xx** | **ошибка сервера** | `500 Internal Server Error`, `502 Bad Gateway`, `503 Service Unavailable` |

Правило: **4xx — разбираемся со своим запросом, 5xx — ждём и пробуем позже**.

## HTTP не хранит состояние

Каждый запрос независим: сервер не помнит, что вы спрашивали секунду назад. Если серверу нужно узнавать клиента (например, после входа на сайт), клиент сам передаёт это в каждом запросе: **cookies** или **токен** в заголовке `Authorization`.

## Версии HTTP

* **HTTP/1.1** — текстовый, тот, что мы видели выше;
* **HTTP/2** — бинарный, несколько запросов в одном соединении;
* **HTTP/3** — то же, но поверх QUIC (UDP).

Для нас как пользователей `requests` смысл не меняется: методы, заголовки и коды те же.

## HTTP — не единственный протокол

Все эти протоколы живут **на одном уровне**: поверх TCP или UDP, поверх IP. Путь «DNS → соединение → шифрование» у них общий, а отличается «разговор» поверх соединения.

| Протокол | Порт | Для чего | Где встретите |
|---|---|---|---|
| **HTTP / HTTPS** | 80 / 443 | запрос → ответ | веб, почти все API |
| **WebSocket** | 80 / 443 | постоянный двусторонний канал | чаты, котировки, совместное редактирование |
| **FTP / SFTP** | 21 / 22 | передача файлов | FTP-серверы NCBI и PubChem для выгрузки баз целиком |
| **SSH** | 22 | удалённый терминал, туннели | вход на вычислительный кластер; на нём же SFTP и SCP |
| **SMTP / IMAP** | 587 / 993 | отправка / получение почты | любой почтовый клиент, уведомления из пайплайнов |
| **MQTT** | 1883 / 8883 | публикация и подписка через брокер | IoT, лабораторные датчики |
| **AMQP** | 5672 | очереди сообщений | RabbitMQ, Celery |
| протоколы баз данных | 5432, 6379, ... | общение с базой | PostgreSQL, Redis: драйверы говорят на них под капотом |

**Почему почти всё идёт через HTTP?** Он универсален: порты 80 и 443 открыты почти в любой сети, клиенты есть для любого языка, отлаживать можно браузером. Поэтому многое туннелируют поверх HTTP, даже когда это не идеально: gRPC, GraphQL, S3, рукопожатие WebSocket.

Практический вывод для нас: нужна одна молекула или сотня — HTTP API. Нужна вся база целиком — FTP или прямые ссылки на дампы, а не миллион запросов к API.

## WebSocket: начинается как HTTP

1. Клиент делает **обычный HTTP-запрос** со специальными заголовками:
   ```
   GET /chat HTTP/1.1
   Upgrade: websocket
   Connection: Upgrade
   ```
2. Сервер отвечает `101 Switching Protocols`, и HTTP на этом соединении заканчивается.
3. **То же самое TCP-соединение** остаётся открытым, и обе стороны шлют сообщения когда угодно.

Главное отличие: в HTTP сервер не может написать первым, а в WebSocket — может.

# Уровень 3: интерфейс (API)

Соединение есть, язык общий. Осталось договориться, **что именно можно спросить и в каком виде придёт ответ**. Это и есть **API** (application programming interface) — тот же интерфейс, что и у классов на прошлой паре, только методы вызываются по сети.

## HTTP и REST — не одно и то же

* **HTTP** — **протокол**: правила обмена сообщениями.
* **REST** — **стиль проектирования интерфейса** поверх HTTP: набор соглашений, как разложить API по URL и методам. Его описал Рой Филдинг в 2000 году (он же один из авторов HTTP).

HTTP прекрасно используется и без REST: обычные веб-страницы, скачивание файлов, RPC, SOAP, GraphQL.

## REST подробно

Основные соглашения:
* **ресурс в URL** — путь описывает «что», а не «что сделать»: `/compound/name/aspirin`, а не `/getCompoundByName`;
* **метод как глагол**: GET — получить, POST — создать, PUT — заменить, DELETE — удалить;
* **статус-коды по назначению**: 404, если ресурса нет; 400, если запрос кривой; 201, если создано;
* **состояние не хранится**: каждый запрос содержит всё нужное;
* **формат ответа** — обычно JSON.

Пример — **PubChem PUG REST**. Путь читается почти как фраза:

```
https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/property/MolecularFormula,MolecularWeight/JSON
                                          └──┬───┘ └─────┬────┘ └────────────────┬───────────────────────┘ └┬─┘
                                          домен    вход: имя            что хотим получить            формат
```

«Соединение, найденное по имени aspirin, — его свойства MolecularFormula и MolecularWeight — в JSON». Заменив последнее слово на `CSV` или `SDF`, получим тот же ответ в другом формате.

### Идемпотентность

Запрос **идемпотентен**, если повторить его несколько раз — то же самое, что выполнить один раз.

* GET, PUT, DELETE — идемпотентны: дважды получить или дважды удалить одно и то же безопасно;
* POST — нет: дважды отправить «создай заказ» — это два заказа.

Это важно для **ретраев**: если ответ не пришёл, идемпотентный запрос можно безопасно повторить, а POST — нет.

### Пагинация

Когда результатов много, API отдаёт их порциями: `?page=2&per_page=100` или `?offset=200&limit=100`. Клиент запрашивает страницы, пока не получит все. Генераторы с прошлых пар для этого идеальны.

### «REST» на практике

Большинство API, которые называют себя REST, следуют не всем соглашениям Филдинга. Честнее их называть «REST-подобными» или «HTTP + JSON API». Поэтому каждый раз читаем документацию конкретного API. Часто она есть в машиночитаемом виде — **OpenAPI** (Swagger).

## Какие ещё бывают интерфейсы

### Синхронные: «спросил — получил ответ»

| Стиль | Поверх чего | Формат | Контракт | Где встречается |
|---|---|---|---|---|
| **REST** | HTTP | обычно JSON | документация, часто OpenAPI | публичные API: PubChem PUG REST, GitHub |
| **RPC** (JSON-RPC, XML-RPC) | HTTP, WebSocket, stdin/stdout | JSON / XML | описание методов | NCBI E-utilities устроены в RPC-стиле; подсказки в PyCharm и VS Code работают через JSON-RPC (протокол LSP) |
| **SOAP** | HTTP (обычно POST) | XML | строгий, WSDL | банки, госсистемы, старые корпоративные сервисы |
| **gRPC** | HTTP/2 | бинарный (Protocol Buffers) | `.proto`-файл, из которого генерируется код | общение микросервисов внутри компаний; быстрый, поддерживает стриминг |
| **GraphQL** | HTTP, один эндпоинт | JSON | типизированная схема | клиент сам выбирает нужные поля; GitHub API v4 |

Для сравнения — тот же поиск в RPC-стиле, NCBI E-utilities. В URL не ресурс, а **название действия** (`esearch`), и все параметры в query string:

```
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pccompound&term=aspirin&retmode=json
```

### Асинхронные: «отправили сообщение, ответ — когда-нибудь или никогда»

| Что | Как устроено | Где встречается |
|---|---|---|
| **Брокеры сообщений** (RabbitMQ / AMQP, Kafka) | сервисы не обращаются друг к другу напрямую, а кладут сообщения в очередь | очереди задач (Celery), обработка событий, длинные вычислительные пайплайны |
| **MQTT** | лёгкая публикация и подписка через брокер | IoT, датчики, лабораторное оборудование |
| **ZeroMQ** | обмен сообщениями без брокера | так Jupyter общается со своим ядром — прямо сейчас, пока вы смотрите этот блокнот |

### Кто начинает разговор

В HTTP разговор всегда начинает клиент. А если серверу нужно сообщить что-то первым?

| Способ | Суть |
|---|---|
| **Polling** | клиент сам спрашивает раз в N секунд: «готово?» |
| **Webhook** | мы даём серверу свой URL, и он сам делает к нам запрос, когда что-то произошло |
| **SSE** (Server-Sent Events) | односторонний поток событий от сервера поверх обычного HTTP |
| **WebSocket** | постоянный двусторонний канал |

### Как выбирают

1. **Насколько строгий нужен контракт?** От «почитайте документацию» (REST) до «код генерируется из схемы» (gRPC, SOAP).
2. **Текст или бинарный формат?** Текст удобно отлаживать, бинарный — быстрее.
3. **Нужен ли ответ сразу?** Синхронный запрос или сообщение в очередь.

Для получения данных из публичных баз почти всегда хватает REST + JSON. Остальное — чтобы знать, что так бывает, и понимать, о чём речь, когда встретите.